check multimodal possibility

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import math
from collections import defaultdict
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# =============================================================================
# Settings
# =============================================================================
BASE_PATH = r"D:\depression_dataset(DAIC-WOZ)"
AUDIO_MODEL_PATH = os.path.join(BASE_PATH, "best_transformer_model.pt")
TEXT_MODEL_PATH = os.path.join(BASE_PATH, "best_text_model.pt")
AUDIO_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_utterance_dataset.pkl")
TEXT_DATA_PATH = os.path.join(BASE_PATH, "preprocessed_text_dataset.pkl")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Optimal thresholds from individual model training
AUDIO_THRESHOLD = 0.68
TEXT_THRESHOLD = 0.20


# =============================================================================
# Model Definition (analyze_dual_attention.py와 동일)
# =============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerDepressionModel(nn.Module):
    """Audio Model"""
    def __init__(self, input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
                 dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32):
        super(TransformerDepressionModel, self).__init__()
        
        self.d_model = d_model
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_wav2vec, batch_ttrs, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_wav2vec.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        combined_features = torch.cat([batch_wav2vec, q_type_embs, ttrs_expanded], dim=1)
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, seq.size(1), device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        x = self.input_projection(padded_sequences)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


class TransformerTextDepressionModel(nn.Module):
    """Text Model"""
    def __init__(self, input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
                 dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32):
        super(TransformerTextDepressionModel, self).__init__()
        
        self.d_model = d_model
        self.q_type_embedding = nn.Embedding(q_type_vocab_size, q_type_embed_dim)
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, batch_bert, batch_ttrs, batch_q_type_ids, num_utterances_list):
        batch_size = len(num_utterances_list)
        device = batch_bert.device
        
        q_type_embs = self.q_type_embedding(batch_q_type_ids)
        ttrs_expanded = batch_ttrs.unsqueeze(1)
        
        combined_features = torch.cat([batch_bert, q_type_embs, ttrs_expanded], dim=1)
        
        participant_sequences = []
        start_idx = 0
        
        for num_utts in num_utterances_list:
            end_idx = start_idx + num_utts
            participant_sequences.append(combined_features[start_idx:end_idx])
            start_idx = end_idx
        
        max_num_utterances = max(num_utterances_list)
        padded_sequences = []
        attention_masks = []
        
        for seq, num_utts in zip(participant_sequences, num_utterances_list):
            if num_utts < max_num_utterances:
                padding = torch.zeros(max_num_utterances - num_utts, seq.size(1), device=device)
                seq = torch.cat([seq, padding], dim=0)
            
            padded_sequences.append(seq)
            
            mask = torch.zeros(max_num_utterances, dtype=torch.bool, device=device)
            mask[num_utts:] = True
            attention_masks.append(mask)
        
        padded_sequences = torch.stack(padded_sequences)
        attention_masks = torch.stack(attention_masks)
        
        x = self.input_projection(padded_sequences)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=device)
        extended_mask = torch.cat([cls_mask, attention_masks], dim=1)
        
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x, src_key_padding_mask=extended_mask)
        
        cls_output = encoded[:, 0, :]
        logits = self.classifier(cls_output)
        
        utterance_features = encoded[:, 1:, :]
        attention_weights = torch.norm(utterance_features, dim=2)
        attention_weights = attention_weights.masked_fill(attention_masks, 0)
        attention_weights = F.softmax(attention_weights + (attention_masks.float() * -1e9), dim=1)
        
        return logits, attention_weights


# =============================================================================
# Data Extraction
# =============================================================================
def extract_predictions_from_models(audio_model, text_model, audio_data, text_data):
    """Extract predictions from both models for all common participants"""
    audio_model.eval()
    text_model.eval()
    
    results = []
    common_pids = set(audio_data.keys()) & set(text_data.keys())
    
    with torch.no_grad():
        for pid in sorted(common_pids):
            audio_info = audio_data[pid]
            text_info = text_data[pid]
            
            label = audio_info['label']
            
            # Audio forward
            audio_utterances = audio_info['utterances']
            wav2vec_features = torch.FloatTensor([utt['wav2vec'] for utt in audio_utterances]).to(DEVICE)
            audio_ttrs = torch.FloatTensor([utt['ttr'] for utt in audio_utterances]).to(DEVICE)
            audio_q_type_ids = torch.LongTensor([utt['q_type_id'] for utt in audio_utterances]).to(DEVICE)
            
            audio_logits, _ = audio_model(
                wav2vec_features, audio_ttrs, audio_q_type_ids, [len(audio_utterances)]
            )
            
            # Text forward
            text_utterances = text_info['utterances']
            bert_features = torch.FloatTensor([utt['bert'] for utt in text_utterances]).to(DEVICE)
            text_ttrs = torch.FloatTensor([utt['ttr'] for utt in text_utterances]).to(DEVICE)
            text_q_type_ids = torch.LongTensor([utt['q_type_id'] for utt in text_utterances]).to(DEVICE)
            
            text_logits, _ = text_model(
                bert_features, text_ttrs, text_q_type_ids, [len(text_utterances)]
            )
            
            audio_prob = torch.sigmoid(audio_logits).item()
            text_prob = torch.sigmoid(text_logits).item()
            
            results.append({
                'pid': pid,
                'label': label,
                'audio_prob': audio_prob,
                'text_prob': text_prob
            })
    
    return results


def split_data_by_metadata_group(results, base_path):
    """Split data into train/val/test based on metadataset.csv groups"""
    # 메타데이터 로드
    meta_df = pd.read_csv(os.path.join(base_path, "metadataset.csv"))
    meta_df['Participant_ID'] = meta_df['Participant_ID'].astype(str)
    
    # PID를 문자열로 변환
    results_df = pd.DataFrame(results)
    results_df['pid'] = results_df['pid'].astype(str)
    
    # 메타데이터와 병합
    results_df = results_df.merge(
        meta_df[['Participant_ID', 'Group']], 
        left_on='pid', 
        right_on='Participant_ID', 
        how='left'
    )
    
    # Group별로 분할
    train_data = results_df[results_df['Group'] == 'Train'].to_dict('records')
    val_data = results_df[results_df['Group'] == 'Validation'].to_dict('records')
    test_data = results_df[results_df['Group'] == 'Test'].to_dict('records')
    
    # 라벨 분포 계산
    def count_labels(data):
        labels = [d['label'] for d in data]
        return labels.count(0), labels.count(1)
    
    train_normal, train_dep = count_labels(train_data)
    val_normal, val_dep = count_labels(val_data)
    test_normal, test_dep = count_labels(test_data)
    
    print(f"\nMetadata-based split:")
    print(f"  Train: {len(train_data)} (Normal: {train_normal}, Depression: {train_dep})")
    print(f"  Val:   {len(val_data)} (Normal: {val_normal}, Depression: {val_dep})")
    print(f"  Test:  {len(test_data)} (Normal: {test_normal}, Depression: {test_dep})")
    
    return train_data, val_data, test_data


# =============================================================================
# Late Fusion with Threshold Correction
# =============================================================================
def evaluate_late_fusion(data, audio_weight, audio_threshold=0.3, text_threshold=0.2):
    """
    Threshold-aware late fusion evaluation
    
    Method 1: Probability fusion (기존 방식)
    Method 2: Decision fusion (threshold 적용 후 voting)
    """
    text_weight = 1 - audio_weight
    
    y_true = []
    y_pred_prob = []
    y_pred_decision = []
    
    for sample in data:
        label = sample['label']
        audio_prob = sample['audio_prob']
        text_prob = sample['text_prob']
        
        y_true.append(label)
        
        # Method 1: Weighted probability fusion
        fused_prob = audio_weight * audio_prob + text_weight * text_prob
        pred_prob = 1 if fused_prob > 0.5 else 0
        y_pred_prob.append(pred_prob)
        
        # Method 2: Decision-level fusion (threshold-aware)
        audio_decision = 1 if audio_prob > audio_threshold else 0
        text_decision = 1 if text_prob > text_threshold else 0
        
        # Weighted voting
        fused_decision = audio_weight * audio_decision + text_weight * text_decision
        pred_decision = 1 if fused_decision > 0.5 else 0
        y_pred_decision.append(pred_decision)
    
    # Calculate metrics
    f1_prob = f1_score(y_true, y_pred_prob, zero_division=0)
    precision_prob = precision_score(y_true, y_pred_prob, zero_division=0)
    recall_prob = recall_score(y_true, y_pred_prob, zero_division=0)
    
    f1_decision = f1_score(y_true, y_pred_decision, zero_division=0)
    precision_decision = precision_score(y_true, y_pred_decision, zero_division=0)
    recall_decision = recall_score(y_true, y_pred_decision, zero_division=0)
    
    # Calculate specificity
    cm_prob = confusion_matrix(y_true, y_pred_prob)
    cm_decision = confusion_matrix(y_true, y_pred_decision)
    
    if cm_prob.shape[0] == 2:
        tn_prob, fp_prob, fn_prob, tp_prob = cm_prob.ravel()
        spec_prob = tn_prob / (tn_prob + fp_prob) if (tn_prob + fp_prob) > 0 else 0
    else:
        spec_prob = 0
    
    if cm_decision.shape[0] == 2:
        tn_decision, fp_decision, fn_decision, tp_decision = cm_decision.ravel()
        spec_decision = tn_decision / (tn_decision + fp_decision) if (tn_decision + fp_decision) > 0 else 0
    else:
        spec_decision = 0
    
    return {
        'prob': {
            'f1': f1_prob, 
            'precision': precision_prob, 
            'recall': recall_prob,
            'specificity': spec_prob
        },
        'decision': {
            'f1': f1_decision, 
            'precision': precision_decision, 
            'recall': recall_decision,
            'specificity': spec_decision
        }
    }


def find_optimal_fusion_weight(train_data, val_data, audio_threshold=0.3, text_threshold=0.2):
    """Find optimal fusion weight using validation set"""
    
    print(f"\n{'='*70}")
    print(f"Late Fusion Weight Optimization (Threshold-Aware)")
    print(f"{'='*70}")
    print(f"Audio Threshold: {audio_threshold}")
    print(f"Text Threshold:  {text_threshold}\n")
    
    weights = np.arange(0, 1.05, 0.05)
    
    results_prob = []
    results_decision = []
    
    for w_audio in weights:
        metrics = evaluate_late_fusion(val_data, w_audio, audio_threshold, text_threshold)
        results_prob.append(metrics['prob'])
        results_decision.append(metrics['decision'])
    
    # Find best weights
    f1_scores_prob = [r['f1'] for r in results_prob]
    f1_scores_decision = [r['f1'] for r in results_decision]
    
    best_idx_prob = np.argmax(f1_scores_prob)
    best_idx_decision = np.argmax(f1_scores_decision)
    
    best_weight_prob = weights[best_idx_prob]
    best_weight_decision = weights[best_idx_decision]
    
    print(f"[Method 1: Probability Fusion]")
    print(f"  Best Audio Weight: {best_weight_prob:.2f}")
    print(f"  Best F1:           {f1_scores_prob[best_idx_prob]:.4f}")
    print(f"  Precision:         {results_prob[best_idx_prob]['precision']:.4f}")
    print(f"  Recall:            {results_prob[best_idx_prob]['recall']:.4f}")
    print(f"  Specificity:       {results_prob[best_idx_prob]['specificity']:.4f}")
    
    print(f"\n[Method 2: Decision Fusion (Threshold-Aware)]")
    print(f"  Best Audio Weight: {best_weight_decision:.2f}")
    print(f"  Best F1:           {f1_scores_decision[best_idx_decision]:.4f}")
    print(f"  Precision:         {results_decision[best_idx_decision]['precision']:.4f}")
    print(f"  Recall:            {results_decision[best_idx_decision]['recall']:.4f}")
    print(f"  Specificity:       {results_decision[best_idx_decision]['specificity']:.4f}")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Method 1
    ax1 = axes[0]
    ax1.plot(weights, f1_scores_prob, 'o-', linewidth=2, markersize=6, color='blue', label='F1')
    ax1.plot(weights, [r['precision'] for r in results_prob], 's--', linewidth=1.5, markersize=4, 
             color='green', alpha=0.7, label='Precision')
    ax1.plot(weights, [r['recall'] for r in results_prob], '^--', linewidth=1.5, markersize=4, 
             color='red', alpha=0.7, label='Recall')
    ax1.plot(weights, [r['specificity'] for r in results_prob], 'd--', linewidth=1.5, markersize=4, 
             color='orange', alpha=0.7, label='Specificity')
    ax1.axvline(best_weight_prob, color='purple', linestyle='--', linewidth=2, 
                label=f'Best: {best_weight_prob:.2f}')
    ax1.set_xlabel('Audio Weight (Text = 1 - Audio)', fontsize=12)
    ax1.set_ylabel('Score', fontsize=12)
    ax1.set_title('Method 1: Probability Fusion', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Method 2
    ax2 = axes[1]
    ax2.plot(weights, f1_scores_decision, 'o-', linewidth=2, markersize=6, color='blue', label='F1')
    ax2.plot(weights, [r['precision'] for r in results_decision], 's--', linewidth=1.5, markersize=4, 
             color='green', alpha=0.7, label='Precision')
    ax2.plot(weights, [r['recall'] for r in results_decision], '^--', linewidth=1.5, markersize=4, 
             color='red', alpha=0.7, label='Recall')
    ax2.plot(weights, [r['specificity'] for r in results_decision], 'd--', linewidth=1.5, markersize=4, 
             color='orange', alpha=0.7, label='Specificity')
    ax2.axvline(best_weight_decision, color='purple', linestyle='--', linewidth=2, 
                label=f'Best: {best_weight_decision:.2f}')
    ax2.set_xlabel('Audio Weight (Text = 1 - Audio)', fontsize=12)
    ax2.set_ylabel('Score', fontsize=12)
    ax2.set_title('Method 2: Decision Fusion (Threshold-Aware)', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    save_path = os.path.join(BASE_PATH, "late_fusion_optimization_corrected.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\nSaved: {save_path}")
    
    return {
        'prob': {'weight': best_weight_prob, 'metrics': results_prob[best_idx_prob]},
        'decision': {'weight': best_weight_decision, 'metrics': results_decision[best_idx_decision]}
    }


def evaluate_on_test(test_data, best_configs, audio_threshold=0.3, text_threshold=0.2):
    """Evaluate best configurations on test set"""
    
    print(f"\n{'='*70}")
    print(f"Test Set Evaluation")
    print(f"{'='*70}\n")
    
    for method_name, config in best_configs.items():
        weight = config['weight']
        metrics = evaluate_late_fusion(test_data, weight, audio_threshold, text_threshold)
        
        # Get the correct method metrics
        if method_name == 'prob':
            test_metrics = metrics['prob']
        else:
            test_metrics = metrics['decision']
        
        print(f"[{method_name.upper()}] Audio Weight: {weight:.2f}")
        print(f"  Test F1:          {test_metrics['f1']:.4f}")
        print(f"  Test Precision:   {test_metrics['precision']:.4f}")
        print(f"  Test Recall:      {test_metrics['recall']:.4f}")
        print(f"  Test Specificity: {test_metrics['specificity']:.4f}\n")
    
    # Baseline comparison
    print(f"[Baseline Comparison]")
    
    # Audio only
    audio_only = evaluate_late_fusion(test_data, 1.0, audio_threshold, text_threshold)
    print(f"  Audio Only:")
    print(f"    F1:          {audio_only['decision']['f1']:.4f}")
    print(f"    Precision:   {audio_only['decision']['precision']:.4f}")
    print(f"    Recall:      {audio_only['decision']['recall']:.4f}")
    print(f"    Specificity: {audio_only['decision']['specificity']:.4f}")
    
    # Text only
    text_only = evaluate_late_fusion(test_data, 0.0, audio_threshold, text_threshold)
    print(f"\n  Text Only:")
    print(f"    F1:          {text_only['decision']['f1']:.4f}")
    print(f"    Precision:   {text_only['decision']['precision']:.4f}")
    print(f"    Recall:      {text_only['decision']['recall']:.4f}")
    print(f"    Specificity: {text_only['decision']['specificity']:.4f}")


# =============================================================================
# Main Execution
# =============================================================================
if __name__ == "__main__":
    print(f"{'='*70}")
    print(f"Threshold-Aware Late Fusion Analysis")
    print(f"{'='*70}\n")
    
    # Load models
    print("Loading audio model...")
    audio_checkpoint = torch.load(AUDIO_MODEL_PATH)
    audio_model = TransformerDepressionModel(
        input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
        dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32
    ).to(DEVICE)
    audio_model.load_state_dict(audio_checkpoint['model_state_dict'])
    audio_model.eval()
    print("Audio model loaded")
    
    print("Loading text model...")
    text_checkpoint = torch.load(TEXT_MODEL_PATH)
    text_model = TransformerTextDepressionModel(
        input_dim=801, d_model=256, nhead=8, num_encoder_layers=3,
        dim_feedforward=512, dropout=0.3, q_type_vocab_size=5, q_type_embed_dim=32
    ).to(DEVICE)
    text_model.load_state_dict(text_checkpoint['model_state_dict'])
    text_model.eval()
    print("Text model loaded")
    
    # Load datasets
    print("\nLoading datasets...")
    with open(AUDIO_DATA_PATH, 'rb') as f:
        audio_data = pickle.load(f)
    with open(TEXT_DATA_PATH, 'rb') as f:
        text_data = pickle.load(f)
    print(f"Audio data: {len(audio_data)} participants")
    print(f"Text data: {len(text_data)} participants")
    
    # Extract predictions
    print("\nExtracting predictions from both models...")
    all_results = extract_predictions_from_models(audio_model, text_model, audio_data, text_data)
    print(f"Extracted predictions for {len(all_results)} participants")
    
    # Split data by metadata groups
    print(f"\nSplitting data by metadata groups...")
    train_data, val_data, test_data = split_data_by_metadata_group(all_results, BASE_PATH)
    
    # Find optimal fusion weights
    best_configs = find_optimal_fusion_weight(train_data, val_data, AUDIO_THRESHOLD, TEXT_THRESHOLD)
    
    # Evaluate on test set
    evaluate_on_test(test_data, best_configs, AUDIO_THRESHOLD, TEXT_THRESHOLD)
    
    print(f"\n{'='*70}")
    print(f"Analysis Complete!")
    print(f"{'='*70}")

nothing noticeable from bert